# 🎬 Movie-Pipeline Mestre Monolítico (Kaggle / Colab / Standalone)

Notebook 100% autossuficiente para execução em nuvem isolada. Contém todas as funções de seleção, download, IA de roteiro ULTRA-DETALHADO (gpt-5-mini com prompts ricos e extensos), inicialização dinâmica do OmniVoice TTS com a voz do narrador Alessandro.mp3, renderização do bloco principal de narração com SLIDESHOW DE 5 A 10 SEGUNDOS POR IMAGEM e MARCA D'ÁGUA DVD BOUNCE ANIMADA E LENTA CORRIGIDA (12s por travessia) e GERADOR INSTANTÂNEO DE VÍDEO LONGO VIA STREAM LOOP (-c copy) QUE CORRESPONDE À DURAÇÃO EXATA DO TXT (DURACAO_MIN) com UPLOAD PARA O GOOGLE DRIVE embutidos diretamente nas células.

---

In [ ]:
# @title 🚀 1. Setup Inicial: Dependências, Secrets e Conexão Google Drive
import os, sys, time, json, re, glob, subprocess, random, shutil, urllib.request, io, math
from pathlib import Path
from dotenv import load_dotenv

# Instala dependências essenciais incluindo omnivoice silenciosamente
os.system("apt-get install -y ffmpeg fonts-dejavu > /dev/null 2>&1")
os.system("pip install python-dotenv requests Pillow pydub openai google-genai edge-tts google-auth google-auth-httplib2 google-api-python-client gradio_client bs4 omnivoice > /dev/null 2>&1")

# Instala a fonte Bungee (usada na watermark animada DVD Bounce)
_bungee_installed = False
try:
    os.system("apt-get install -y fonts-bungee > /dev/null 2>&1")
    # Fallback: Download direto do Google Fonts se apt não tiver a fonte
    import subprocess as _sp
    _fc = _sp.run(['fc-list', ':family=Bungee'], capture_output=True, text=True)
    if 'Bungee' in _fc.stdout:
        _bungee_installed = True
        print('✅ Fonte Bungee encontrada no sistema!')
    else:
        print('⬇️ Baixando fonte Bungee do Google Fonts...')
        os.makedirs('/usr/share/fonts/truetype/bungee', exist_ok=True)
        urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/bungee/Bungee-Regular.ttf', '/usr/share/fonts/truetype/bungee/Bungee-Regular.ttf')
        os.system('fc-cache -f > /dev/null 2>&1')
        _fc2 = _sp.run(['fc-list', ':family=Bungee'], capture_output=True, text=True)
        _bungee_installed = 'Bungee' in _fc2.stdout
        print(f"{'✅ Fonte Bungee instalada!' if _bungee_installed else '⚠️ Bungee não disponível, usando DejaVu Sans Bold como fallback.'}")
except Exception as _fe:
    print(f'⚠️ Erro ao instalar fonte Bungee (fallback DejaVu Sans Bold): {_fe}')

WATERMARK_FONT = 'Bungee' if _bungee_installed else 'DejaVu Sans'
print(f'🔤 Fonte da Watermark: {WATERMARK_FONT}')

# --- 1. CARREGAMENTO DE SEGREDOS / KEYS ---
def _ks(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except:
        load_dotenv()
        return os.getenv(name, "")

TMDB_API_KEY            = _ks("TMDB_API_KEY")
GEMINI_API_KEY          = _ks("GEMINI_API_KEY")
OPENAI_API_KEY          = _ks("OPENAI_API_KEY")
DEEPSEEK_API_KEY        = _ks("DEEPSEEK_API_KEY")
AZURE_OPENAI_ENDPOINT   = _ks("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY    = _ks("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = _ks("AZURE_OPENAI_DEPLOYMENT") or "gpt-5-mini"
DRIVE_ACCESS_TOKEN      = _ks("DRIVE_ACCESS_TOKEN")
DRIVE_REFRESH_TOKEN     = _ks("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID         = _ks("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET     = _ks("DRIVE_CLIENT_SECRET")

# --- 2. CONEXÃO GOOGLE DRIVE ---
drive_service = None
try:
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
    
    creds = Credentials(
        token=DRIVE_ACCESS_TOKEN if DRIVE_ACCESS_TOKEN else None,
        refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID,
        client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"]
    )
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=creds)
    print("✅ Google Drive Autenticado com sucesso!")
except Exception as e:
    print(f"⚠️ Executando sem Google Drive: {e}")

# --- 3. FUNÇÕES AUXILIARES DO GOOGLE DRIVE ---
def _buscar_id(caminho_no_drive):
    if not drive_service: return None
    partes = caminho_no_drive.strip("/").split("/")
    parent_id = "root"
    for parte in partes:
        query = f"name='{parte}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id, mimeType)").execute()
        arquivos = results.get("files", [])
        if not arquivos: return None
        parent_id = arquivos[0]["id"]
    return parent_id

def _garantir_pasta(caminho_pasta):
    if not drive_service: return None
    partes = caminho_pasta.strip("/").split("/")
    parent_id = "root"
    for pasta in partes:
        query = f"name='{pasta}' and '{parent_id}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        if existentes:
            parent_id = existentes[0]["id"]
        else:
            nova = drive_service.files().create(
                body={"name": pasta, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
                fields="id"
            ).execute()
            parent_id = nova["id"]
    return parent_id

def baixar_do_drive(caminho_no_drive, destino_local):
    if not drive_service or os.path.exists(destino_local): return True
    try:
        file_id = _buscar_id(caminho_no_drive)
        if not file_id: return False
        os.makedirs(os.path.dirname(destino_local), exist_ok=True)
        request = drive_service.files().get_media(fileId=file_id)
        with open(destino_local, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()
        return True
    except: return False

def baixar_pasta_do_drive(caminho_pasta_drive, pasta_destino_local):
    if not drive_service: return []
    try:
        folder_id = _buscar_id(caminho_pasta_drive)
        if not folder_id: return []
        os.makedirs(pasta_destino_local, exist_ok=True)
        results = drive_service.files().list(q=f"'{folder_id}' in parents and trashed=false", fields="files(id, name)").execute()
        arquivos = results.get("files", [])
        baixados = []
        for arq in arquivos:
            dest = os.path.join(pasta_destino_local, arq["name"])
            if baixar_do_drive(f"{caminho_pasta_drive}/{arq['name']}", dest):
                baixados.append(dest)
        return baixados
    except Exception as e:
        print(f"Aviso download pasta Drive: {e}")
        return []

def salvar_no_drive(caminho_local, caminho_destino_drive):
    if not drive_service or not os.path.exists(caminho_local): return
    try:
        partes = caminho_destino_drive.strip("/").split("/")
        nome_arquivo = partes[-1]
        pasta_drive  = "/".join(partes[:-1]) if len(partes) > 1 else ""
        parent_id = _garantir_pasta(pasta_drive) if pasta_drive else "root"
        query = f"name='{nome_arquivo}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        media = MediaFileUpload(caminho_local, resumable=True)
        if existentes:
            drive_service.files().update(fileId=existentes[0]["id"], media_body=media).execute()
        else:
            drive_service.files().create(
                body={"name": nome_arquivo, "parents": [parent_id]},
                media_body=media, fields="id"
            ).execute()
        print(f"  ☁️ Salvo no Drive: {caminho_destino_drive}")
    except Exception as e:
        print(f"  ⚠️ Erro ao salvar {caminho_destino_drive}: {e}")

print("✨ Setup Inicial Concluído!")


In [ ]:
# @title 📌 2. Seleção do Filme e Leitura de Metadados (Drive ou TMDB API)
import requests, re

def generate_movie_slug(title: str, release_date: str = "") -> str:
    year = release_date[:4] if release_date and len(release_date) >= 4 else ""
    clean = re.sub(r'[^\w\s]', '', title.lower())
    slug_base = "_".join(clean.split())
    if year and year not in slug_base:
        return f"{slug_base}_{year}"
    return slug_base

def get_movie_project(language="pt-BR"):
    # 1. Tenta carregar do Google Drive se já houver um projeto em Movie-Pipeline/Projetos/
    if drive_service:
        try:
            proj_folder_id = _buscar_id("Movie-Pipeline/Projetos")
            if proj_folder_id:
                res = drive_service.files().list(q=f"'{proj_folder_id}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'", fields="files(id, name)").execute()
                folders = res.get("files", [])
                if folders:
                    slug = folders[0]["name"]
                    print(f"📦 Encontrado projeto pendente no Drive: '{slug}'")
                    txt_name = f"{slug}.txt"
                    txt_local = f"temp/{txt_name}"
                    if baixar_do_drive(f"Movie-Pipeline/Projetos/{slug}/{txt_name}", txt_local):
                        with open(txt_local, "r", encoding="utf-8") as f:
                            content = f.read()
                        title_m = re.search(r"TITULO:\s*(.+)", content)
                        overview_m = re.search(r"SINOPSE:\s*(.+)", content)
                        tmdb_id_m = re.search(r"TMDB_ID:\s*(.+)", content)
                        duracao_m = re.search(r"(?:DURACAO_MIN|DURACAO_MINUTOS|DURACAO|RUNTIME):\s*(\d+)", content, re.IGNORECASE)
                        
                        title = title_m.group(1).strip() if title_m else slug
                        overview = overview_m.group(1).strip() if overview_m else ""
                        tmdb_id = tmdb_id_m.group(1).strip() if tmdb_id_m else None
                        runtime = int(duracao_m.group(1)) if duracao_m else 110
                        
                        info = {
                            "title": title,
                            "original_title": title,
                            "slug": slug,
                            "overview": overview,
                            "runtime": runtime,
                            "txt_path": txt_local
                        }
                        if tmdb_id: info["tmdb_id"] = tmdb_id
                        return info
        except Exception as e: print(f"Aviso busca Drive: {e}")

    # 2. Fallback: Busca filme em alta na API do TMDB
    if not TMDB_API_KEY:
        print("⚠️ TMDB_API_KEY não encontrada nos segredos enviadas.")
        return None
    url = "https://api.themoviedb.org/3/trending/movie/day"
    params = {"api_key": TMDB_API_KEY, "language": language}
    res = requests.get(url, params=params)
    if res.status_code != 200:
        print(f"⚠️ Erro ao consultar TMDB API ({res.status_code}): {res.text[:100]}")
        return None
    results = res.json().get("results", [])
    if not results: return None
    
    item = results[0]
    tmdb_id = item.get("id")
    title = item.get("title") or item.get("name")
    det_res = requests.get(f"https://api.themoviedb.org/3/movie/{tmdb_id}", params=params).json()
    
    slug = generate_movie_slug(title, det_res.get("release_date", ""))
    runtime = det_res.get("runtime", 110)
    info = {
        "tmdb_id": tmdb_id,
        "title": title,
        "original_title": det_res.get("original_title", title),
        "slug": slug,
        "overview": det_res.get("overview") or item.get("overview"),
        "release_date": det_res.get("release_date", ""),
        "runtime": runtime,
        "genres": [g.get("name") for g in det_res.get("genres", [])]
    }
    os.makedirs("temp", exist_ok=True)
    txt_file = f"temp/{slug}.txt"
    content = f"TITULO: {info['title']}\nSLUG: {slug}\nTMDB_ID: {tmdb_id}\nDURACAO_MIN: {runtime}\nLANCAMENTO: {info['release_date']}\nSINOPSE: {info['overview']}"
    with open(txt_file, "w", encoding="utf-8") as f: f.write(content)
    info["txt_path"] = txt_file
    return info

filme = get_movie_project()
if filme:
    print(f"🎬 Filme Selecionado: {filme['title']}")
    print(f"🔑 Slug: {filme['slug']}")
    print(f"⏱️ Duração Oficial do Título (do TXT): {filme.get('runtime', 110)} minutos")
    print(f"📄 TXT Gerado/Carregado: {filme['txt_path']}")
else:
    print("⚠️ Nenhum filme selecionado.")


In [ ]:
# @title 🖼️ 3. Download & Filtro de Imagens (Apenas 16:9 / 1:1, Min 30 - Max 150)
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
import urllib.parse

def is_valid_aspect_ratio(w, h):
    if not w or not h: return False
    return (w / h) >= 0.95 # Aceita apenas 1:1 ou 16:9 (descarta 9:16 e 3:4)

def download_and_verify_image(url, save_path):
    try:
        res = requests.get(url, timeout=10)
        if res.status_code == 200:
            img = Image.open(BytesIO(res.content))
            if is_valid_aspect_ratio(img.width, img.height):
                img.convert("RGB").save(save_path, "JPEG")
                return True
    except: pass
    return False

def download_images_for_movie(movie_info, min_images=30, max_images=150):
    slug = movie_info["slug"]
    img_dir = f"temp/{slug}/imagens"
    os.makedirs(img_dir, exist_ok=True)
    
    # 1. Tenta carregar do Google Drive se a pasta de imagens já existir no Drive
    if drive_service:
        drive_imgs = baixar_pasta_do_drive(f"Movie-Pipeline/Projetos/{slug}/imagens", img_dir)
        if len(drive_imgs) >= min_images:
            print(f"🖼️ Baixadas {len(drive_imgs)} imagens existentes do Google Drive para o vídeo.")
            return drive_imgs
            
    urls = []
    if "tmdb_id" in movie_info and TMDB_API_KEY:
        res = requests.get(f"https://api.themoviedb.org/3/movie/{movie_info['tmdb_id']}/images", params={"api_key": TMDB_API_KEY}).json()
        for bg in res.get("backdrops", []) + res.get("stills", []):
            if is_valid_aspect_ratio(bg.get("width",0), bg.get("height",0)):
                urls.append(f"https://image.tmdb.org/t/p/original{bg['file_path']}")
            
    urls = list(dict.fromkeys(urls))
    saved = []
    for idx, u in enumerate(urls[:max_images]):
        p = f"{img_dir}/img_{idx+1:03d}.jpg"
        if download_and_verify_image(u, p):
            saved.append(p)
            
    # Fallback se < 30
    if len(saved) < min_images:
        print(f"Obtidas {len(saved)} imagens no TMDB. Rodando fallback scraper...")
        q = f"{movie_info['title']} movie wallpaper 16:9 HD"
        html = requests.get(f"https://html.duckduckgo.com/html/?q={urllib.parse.quote(q)}", headers={"User-Agent":"Mozilla/5.0"}).text
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            if "uddg=" in a["href"] and len(saved) < min_images:
                parsed = urllib.parse.parse_qs(urllib.parse.urlparse(a["href"]).query)
                if "uddg" in parsed:
                    p = f"{img_dir}/img_{len(saved)+1:03d}.jpg"
                    if download_and_verify_image(parsed["uddg"][0], p):
                        saved.append(p)
                        
    print(f"🖼️ Total de {len(saved)} imagens salvas para o vídeo.")
    return saved

if 'filme' in locals() and filme:
    imagens = download_images_for_movie(filme)
    if drive_service:
        print("Subindo imagens pro Google Drive...")
        for img in imagens:
            salvar_no_drive(img, f"Movie-Pipeline/Projetos/{filme['slug']}/imagens/{os.path.basename(img)}")


In [ ]:
# @title 📝 4. Fábrica de Roteiro IA ULTRA-DETALHADO (Azure -> Gemini -> DeepSeek -> OpenAI)
def generate_llm_text(prompt, system_instruction=""):
    # 1. Azure OpenAI (gpt-5-mini)
    if AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT:
        try:
            from openai import OpenAI
            az_cli = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)
            deployment_name = AZURE_OPENAI_DEPLOYMENT or "gpt-5-mini"
            res = az_cli.chat.completions.create(
                model=deployment_name,
                messages=[{"role":"system","content":system_instruction},{"role":"user","content":prompt}]
            )
            text = res.choices[0].message.content.strip()
            if text: print("✔ Gerado via Azure OpenAI"); return text
        except Exception as e: print(f"  [Azure Fail]: {e}")
        
    # 2. Gemini (4 Modelos)
    if GEMINI_API_KEY:
        try:
            from google import genai
            g_cli = genai.Client(api_key=GEMINI_API_KEY)
            for m in ["gemini-3.5-flash", "gemini-3.1-pro-preview", "gemini-3.1-flash-lite", "gemini-2.5-pro"]:
                try:
                    resp = g_cli.models.generate_content(model=m, contents=f"{system_instruction}\n\n{prompt}")
                    if resp and resp.text:
                        print(f"✔ Gerado via Gemini ({m})"); return resp.text.strip()
                except Exception as ge: print(f"  [Gemini {m} Fail]: {ge}")
        except Exception as e: print(f"  [Gemini Client Fail]: {e}")
        
    # 3. DeepSeek
    if DEEPSEEK_API_KEY:
        try:
            from openai import OpenAI
            ds_cli = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
            res = ds_cli.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role":"system","content":system_instruction},{"role":"user","content":prompt}]
            )
            text = res.choices[0].message.content.strip()
            if text: print("✔ Gerado via DeepSeek"); return text
        except Exception as e: print(f"  [DeepSeek Fail]: {e}")
        
    # 4. OpenAI (gpt-5-mini)
    if OPENAI_API_KEY:
        try:
            from openai import OpenAI
            oai_cli = OpenAI(api_key=OPENAI_API_KEY)
            res = oai_cli.chat.completions.create(
                model="gpt-5-mini",
                messages=[{"role":"system","content":system_instruction},{"role":"user","content":prompt}]
            )
            text = res.choices[0].message.content.strip()
            if text: print("✔ Gerado via OpenAI (gpt-5-mini)"); return text
        except Exception as e: print(f"  [OpenAI Fail]: {e}")
        
    raise RuntimeError("Todas as IAs da cadeia falharam!")

def generate_detailed_movie_script(movie_info):
    sys_prompt = (
        "Você é o narrador oficial de um canal premium de resumos e reviews detalhadas de cinema. "
        "Sua missão é escrever um roteiro EXTREMAMENTE DETALHADO, MAJESTOSO E EXTENSO, descrevendo cena por cena com riqueza de detalhes, diálogos marcantes, contexto psicológico e tensão máxima. "
        "Seja o mais longo e detalhado possível. Não economize palavras nem resuma nada de forma rasa. Escreva textos longos, profundos e extremamente minuciosos. "
        "Não use marcações de áudio como '[som de suspense]' ou '(música sobe)'. Escreva apenas o texto corrido a ser narrado."
    )
    title = movie_info["title"]
    
    p1 = f"Filme: {title}\nSinopse: {movie_info['overview']}\nEscreva a PARTE 1 extremamente longa e minuciosa (Introdução envolvente, gancho inicial marcante, contexto profundo da história e apresentação completa dos protagonists e do cenário)."
    part1 = generate_llm_text(p1, sys_prompt)
    
    p2 = f"Filme: {title}\nContinuação da Parte 1:\n{part1[-400:]}\nEscreva a PARTE 2 extremamente longa e detalhada (Desenvolvimento aprofundado da trama, arcos dos personagens, segredos revelados e construção do conflito central cena por cena)."
    part2 = generate_llm_text(p2, sys_prompt)
    
    p3 = f"Filme: {title}\nContinuação da Parte 2:\n{part2[-400:]}\nEscreva a PARTE 3 extremamente longa e intensa (Desenvolvimento do clímax, momentos de maior tensão, reviravoltas chocantes e sequências de impacto)."
    part3 = generate_llm_text(p3, sys_prompt)
    
    p4 = f"Filme: {title}\nContinuação da Parte 3:\n{part3[-400:]}\nEscreva a PARTE 4 extremamente completa (Conclusão emocionante, resolução de todos os arcos dos personagens, explicação das mensagens centrais e desfecho magistral)."
    part4 = generate_llm_text(p4, sys_prompt)
    
    return f"{part1}\n\n{part2}\n\n{part3}\n\n{part4}"

if 'filme' in locals() and filme:
    roteiro = generate_detailed_movie_script(filme)
    print(f"📝 Roteiro Gerado: {len(roteiro)} caracteres.")


In [ ]:
# @title 🎙️ 5. Fábrica de Áudio Omni TTS (Servidores + Síntese com Clonagem Alessandro.mp3)
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment
import subprocess, urllib.request, time, os, torch

# --- 1. DOWNLOAD DA VOZ DE CLONAGEM (Alessandro.mp3) DO GOOGLE DRIVE ---
REF_CLONE_PATH = "temp/clonagem_ref.mp3"
os.makedirs("temp", exist_ok=True)

if drive_service:
    clonagem_id = _buscar_id("Movie-Pipeline/Assets/Clonagem")
    if clonagem_id:
        res = drive_service.files().list(q=f"'{clonagem_id}' in parents and trashed=false", fields="files(id, name)").execute()
        files = res.get("files", [])
        if files:
            nome_ref = files[0]['name']
            print(f"🎙️ Baixando arquivo de clonagem '{nome_ref}' do Drive...")
            baixar_do_drive(f"Movie-Pipeline/Assets/Clonagem/{nome_ref}", REF_CLONE_PATH)
            print(f"✅ Áudio de clonagem '{nome_ref}' pronto em {REF_CLONE_PATH}!")

# --- 2. INICIALIZAÇÃO DINÂMICA DOS SERVIDORES OMNIVOICE --- 
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"🚀 Inicializando Servidores OmniVoice para o TTS (GPUs disponíveis: {GPU_COUNT})...")

_server_procs = []
OMNIVOICE_PORTS = []

for gpu_idx in range(min(GPU_COUNT, 2)):
    port = 8001 + gpu_idx
    log = open(f"temp/omnivoice_gpu{gpu_idx}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_idx)}
    proc = subprocess.Popen(
        ["omnivoice-demo", "--ip", "127.0.0.1", "--port", str(port)],
        env=env, stdout=log, stderr=subprocess.STDOUT
    )
    _server_procs.append(proc)
    OMNIVOICE_PORTS.append(port)
    print(f"  GPU{gpu_idx} (CUDA:{gpu_idx}) → Servidor OmniVoice na porta {port} iniciando...")

def _aguardar_servidor(port, label):
    for i in range(25):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2)
            print(f"  ✅ {label} online na porta {port}!")
            return True
        except:
            time.sleep(2)
    print(f"  ⚠️ Servidor na porta {port} subindo em segundo plano...")
    return False

for idx, p in enumerate(OMNIVOICE_PORTS):
    _aguardar_servidor(p, f"OmniVoice #{idx+1}")

# --- 3. SÍNTESE PARALELA E CONCATENAÇÃO ---
def split_text_into_two(text):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    mid = max(1, len(paras)//2)
    return "\n\n".join(paras[:mid]), "\n\n".join(paras[mid:])

def synthesize_omnivoice_block(block_idx, text, port=8001):
    out_wav = f"temp/audio_block_{block_idx}.wav"
    if not os.path.exists(REF_CLONE_PATH):
        raise RuntimeError(f"Áudio de referência para clonagem não encontrado em {REF_CLONE_PATH}!")
        
    from gradio_client import Client, handle_file
    print(f"🎙️ Sintetizando Bloco {block_idx} no OmniVoice (Porta {port}) com clonagem 'Alessandro.mp3'...")
    
    cli = Client(f"http://127.0.0.1:{port}/")
    try:
        res = cli.predict(
            text=text,
            lang="Portuguese",
            ref_aud=handle_file(REF_CLONE_PATH),
            ref_text="",
            instruct="male, young adult",
            ns=32.0, gs=2.0, dn=True, sp=1.0, du=0.0,
            pp=True, po=True,
            api_name="/_clone_fn"
        )
    except Exception as e:
        print(f"  ⚠️ Endpoint /_clone_fn fallback: {e}")
        res = cli.predict(handle_file(REF_CLONE_PATH), text, api_name="/predict")
        
    res_path = res[0] if isinstance(res, (list, tuple)) else res
    if res_path and os.path.exists(res_path):
        AudioSegment.from_file(res_path).export(out_wav, format="wav")
        print(f"  ✅ Bloco {block_idx} sintetizado com SUCESSO pelo OmniVoice!")
        return out_wav
    else:
        raise RuntimeError(f"Falha na síntese do Bloco {block_idx} pelo OmniVoice na porta {port}.")

def generate_voiceover_parallel(script_text):
    t1, t2 = split_text_into_two(script_text)
    p1 = OMNIVOICE_PORTS[0] if len(OMNIVOICE_PORTS) > 0 else 8001
    p2 = OMNIVOICE_PORTS[1] if len(OMNIVOICE_PORTS) > 1 else 8002
    
    with ThreadPoolExecutor(max_workers=2) as ex:
        f1 = ex.submit(synthesize_omnivoice_block, 1, t1, p1)
        f2 = ex.submit(synthesize_omnivoice_block, 2, t2, p2)
        b1, b2 = f1.result(), f2.result()
        
    final_wav = "temp/narracao_final.wav"
    a1, a2 = AudioSegment.from_file(b1), AudioSegment.from_file(b2)
    combined = a1 + AudioSegment.silent(duration=400) + a2
    combined.export(final_wav, format="wav")
    return final_wav

if 'roteiro' in locals() and roteiro:
    audio_narracao = generate_voiceover_parallel(roteiro)
    print(f"🎙️ Narração final por clonagem OmniVoice gerada em: {audio_narracao}")


In [ ]:
# @title 🎥 6. Renderização Estável de Slideshow (5s a 10s por imagem) + Watermark ASS Animada e Lenta (12s) + Stream Loop (-c copy)
def ensure_dvd_bounce_30min_ass(ass_output_path="temp/render/watermark_30min.ass"):
    os.makedirs(os.path.dirname(ass_output_path), exist_ok=True)
    font = WATERMARK_FONT  # Bungee ou DejaVu Sans (fallback garantido)
    header = f"[Script Info]\nScriptType: v4.00+\nPlayResX: 1920\nPlayResY: 1080\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\nStyle: DVDBounce,{font},32,&H4DFFFFFF,&H4DFFFFFF,&H4D000000,&H4D000000,-1,0,0,0,100,100,0,0,1,2,0,5,10,10,10,1\nStyle: DVDBounceSub,{font},22,&H4DFFFFFF,&H4DFFFFFF,&H4D000000,&H4D000000,0,0,0,0,100,100,0,0,1,1,0,5,10,10,10,1\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
    positions = [(200,150), (1720,250), (1720,930), (200,880), (960,150), (200,540)]
    events = []
    t, pos_idx, step = 0.0, 0, 12.0 # ANIMAÇÃO DVD BOUNCE LENTA: 12 segundos por travessia!
    limit_30min = 1800.0
    def _fmt(s):
        return f"{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s-int(s))*100):02d}"
    
    while t < limit_30min:
        t_end = min(t + step, limit_30min)
        p1, p2 = positions[pos_idx % len(positions)], positions[(pos_idx + 1) % len(positions)]
        move_tag = f"\\move({p1[0]},{p1[1]},{p2[0]},{p2[1]})"
        text_main = "ASSISTA COMPLETO NO TELEGRAM -> @LehDramas"
        text_sub = "(Link Direto no 1º Comentário Fixado)"
        ass_newline = chr(92) + 'N'  # Gera literal \N para o ASS sem conflitar com Python Unicode escape
        # Tag \move aplicada uma única vez no início do diálogo ASS
        events.append(f"Dialogue: 0,{_fmt(t)},{_fmt(t_end)},DVDBounce,,0,0,0,,{{{move_tag}}}{text_main}{ass_newline}{text_sub}")
        t = t_end
        pos_idx += 1
        
    with open(ass_output_path, "w", encoding="utf-8") as f: f.write(header + "\n".join(events))
    return ass_output_path

def render_long_movie_video(slug, images, voiceover_path, output_dir="output", target_runtime_minutes=110):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs("temp/render", exist_ok=True)
    
    final_mp4 = f"{output_dir}/{slug}.mp4"
    slide_mp4 = "temp/render/slideshow.mp4"
    ass_path  = ensure_dvd_bounce_30min_ass()
    intro_local = "temp/intro.mp4"
    intro_norm  = "temp/render/intro_norm.mp4"
    
    # 1. Duração do bloco de narração
    dur = len(AudioSegment.from_file(voiceover_path)) / 1000.0
    
    # 2. Gera a lista de imagens com duração VARIANDO DE 5.0s A 10.0s POR FOTO conforme pedido pelo usuário
    concat_txt = "temp/render/slideshow_list.txt"
    lines, curr = [], 0.0
    while curr < dur:
        random.shuffle(images)
        for img in images:
            d = random.uniform(5.0, 10.0)
            lines.append(f"file '{os.path.abspath(img).replace('\\','/')}'\nduration {d:.2f}")
            curr += d
            if curr >= dur: break
    lines.append(f"file '{os.path.abspath(images[0]).replace('\\','/')}'")
    with open(concat_txt, "w", encoding="utf-8") as f: f.write("\n".join(lines))
    
    ass_esc = os.path.abspath(ass_path).replace("\\", "/").replace(":", "\\:")
    
    # 3. FILTRO DE VÍDEO ULTRA-ESTÁVEL (Sem zoompan para estabilidade total de tempo, resolução 1920x1080 + pad 16:9 + watermark animada .ass)
    vf = f"scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:black,ass='{ass_esc}'"
    
    print(f"🚀 1/3 Renderizando bloco do filme ({dur:.1f}s = {dur/60:.1f}min) COM SLIDESHOW (5s a 10s por imagem) + MARCA D'ÁGUA ANIMADA LENTA...")
    cmd_render = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", concat_txt, "-i", voiceover_path, "-vf", vf, "-c:v", "libx264", "-preset", "ultrafast", "-threads", "0", "-pix_fmt", "yuv420p", "-r", "30", "-c:a", "aac", "-ar", "44100", "-ac", "2", "-b:a", "192k", "-shortest", slide_mp4]
    subprocess.run(cmd_render, check=True)
        
    # 4. Baixa e Normaliza a Intro UMA ÚNICA VEZ
    has_intro = baixar_do_drive("Movie-Pipeline/Assets/intro.mp4", intro_local)
    if has_intro and os.path.exists(intro_local):
        print("🎬 2/3 Normalizando Vinheta Intro...")
        cmd_intro = [
            "ffmpeg", "-y", "-i", intro_local,
            "-vf", "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:black",
            "-r", "30", "-c:v", "libx264", "-preset", "ultrafast", "-pix_fmt", "yuv420p",
            "-c:a", "aac", "-ar", "44100", "-ac", "2", "-b:a", "192k",
            intro_norm
        ]
        subprocess.run(cmd_intro, check=True)
    
    # 5. REPETIÇÃO EM LOOP INSTANTÂNEO (-c copy) PARA A DURAÇÃO EXATA DO TXT (DURACAO_MIN)
    target_sec = target_runtime_minutes * 60.0
    num_loops = math.ceil(target_sec / dur)
    total_dur_min = (dur * num_loops) / 60.0
    print(f"⚡ 3/3 Duração do Título do TXT: {target_runtime_minutes} min. Gerando vídeo de {total_dur_min:.1f} minutos ({num_loops}x repetições) via concat -c copy...")
    
    concat_final_list = "temp/render/concat_stream_list.txt"
    concat_lines = []
    if os.path.exists(intro_norm):
        concat_lines.append(f"file '{os.path.abspath(intro_norm).replace('\\','/')}'")
        
    slide_abs = os.path.abspath(slide_mp4).replace('\\','/')
    for _ in range(num_loops):
        concat_lines.append(f"file '{slide_abs}'")
        
    with open(concat_final_list, "w", encoding="utf-8") as f:
        f.write("\n".join(concat_lines))
        
    cmd_fast_concat = [
        "ffmpeg", "-y", "-f", "concat", "-safe", "0",
        "-i", concat_final_list,
        "-c", "copy",
        final_mp4
    ]
    subprocess.run(cmd_fast_concat, check=True)
    
    print(f"🎉 Vídeo Finalizado com Sucesso ({target_runtime_minutes} min): {final_mp4} ({total_dur_min:.1f} min no total com {num_loops}x repetições e marca d'água animada legível!)")
    return final_mp4

if 'audio_narracao' in locals() and os.path.exists(audio_narracao):
    target_min = filme.get('runtime', 110) if 'filme' in locals() and filme else 110
    video_final = render_long_movie_video(filme['slug'], imagens, audio_narracao, target_runtime_minutes=target_min)
    
    # 6. UPLOAD AUTOMÁTICO PARA O GOOGLE DRIVE
    if drive_service:
        print(f"☁️ Fazendo upload do vídeo final '{os.path.basename(video_final)}' ({target_min} min) para o Google Drive...")
        salvar_no_drive(video_final, f"Movie-Pipeline/Projetos/{filme['slug']}/{os.path.basename(video_final)}")
        salvar_no_drive(video_final, f"Movie-Pipeline/Resultados/{os.path.basename(video_final)}")
        print("🚀 UPLOAD CONCLUÍDO COM SUCESSO NO GOOGLE DRIVE!")
